In [2]:
import os
# 攻击者受害者配对
dataset_name = "lfw"
dataset_dir = r'./eval/lfw-112'
dataset_txt = r'./eval/lfw-select-aligned-label.txt'

# 分隔 前10为目标人脸，11-100为原始人脸  同AT3D与SiblingAttack
attack_img_paths = []
victim_img_paths = []
with open(dataset_txt) as fin:
    img_names = fin.readlines()
    for idx, img_name in enumerate(img_names):
        img_path = os.path.join(dataset_dir, img_name.strip())
        if idx < 10:
            victim_img_paths.append(img_path)
        else:
            attack_img_paths.append(img_path)

In [1]:
# 生成AdvFaceGAN的对抗样本
from PIL import Image
import torch
import torchvision
from AdvFaceGAN import Generator
from tqdm import tqdm
from torchvision.utils import save_image

# 指定模型
model_dir = "target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15"
epoch_id = 2490

# 加载预训练模型
fake_generator = Generator(is_target=True).eval().cuda()
model_generator_dict = torch.load('./save_dir/' + model_dir + '/model/' + '%05d_generator.pth' % epoch_id, weights_only=True)
fake_generator.load_state_dict(model_generator_dict)
test_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize((112,112)),
    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
    torchvision.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5], inplace=True),  # 归一化
])
# 保存路径
save_dir = './data/' +f"AdvFaceGAN_{model_dir} {epoch_id}_{dataset_name}_eps5_tpert4.4"
os.makedirs(save_dir, exist_ok=True)
with torch.no_grad():
    # 对每对图片生成对抗样本
    for attack_img_path in tqdm(attack_img_paths, desc="Outer loop (attack images)"):
        for victim_img_path in victim_img_paths:
            attack_img = test_transforms(Image.open(attack_img_path).convert('RGB')).unsqueeze(0).cuda()
            victim_img = test_transforms(Image.open(victim_img_path).convert('RGB')).unsqueeze(0).cuda()
            # Perform Attack
            _,adv_attack_img = fake_generator(sources=attack_img, targets=victim_img)
            # Save images
            out_dir = f"{save_dir}/{os.path.basename(victim_img_path).split('.')[0]}+{os.path.basename(attack_img_path).split('.')[0]}"
            os.makedirs(out_dir,exist_ok=True)
            # 保存图片
            save_image(adv_attack_img*0.5+0.5, out_dir+"/adv.png")
            save_image(attack_img*0.5+0.5, out_dir+"/source.png")
            save_image(victim_img*0.5+0.5, out_dir+"/target.png")

NameError: name 'dataset_name' is not defined

In [2]:
# WhiteBox evaluation
import torch
import torchvision
from PIL import Image
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os
import numpy as np
    
models_info = {}
test_whitebox_model_name_list = ['ArcFace','FaceNet-casia','ResNet50']
asr1_white=np.empty((len(test_whitebox_model_name_list), 12))
asr2_white=np.empty((len(test_whitebox_model_name_list), 12))
with torch.no_grad():
    for idxx, model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------start white evaluate frmodel {0}-------------------\n".format(model_name))
        th = threshold_lfw[model_name]['cos']
        model, img_shape = getmodel(model_name)
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/TIPIM_{model_name}",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy, adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------start white evaluate target method {0}-------------------\n".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for idx,adv_pair in tqdm(enumerate(subdirectories)):
                test_transforms = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                # 计算视觉指标 
                # 扰动二范数 统一到112*112
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
            print(model_name, " benchmark threshold_lfw:%f" % threshold_lfw[model_name]['cos'])
            print(model_name, " benchmark rate:%f" % threshold_lfw[model_name]['cos_acc'])
            print(model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(model_name, " true pert:%f" % np.mean(true_perts_score))
            print(model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(model_name, " average mse:%f" % np.mean(mse_scores))
            print(model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))
            asr1_white[idxx][idxy]=np.mean(np.array(target_simi_scores) > th)
            asr2_white[idxx][idxy]=np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th))

-----------------start white evaluate frmodel ArcFace-------------------

Load existing checkpoint
-----------------start white evaluate target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------



0it [00:00, ?it/s]C:\yy\installed_software\Anaconda3\envs\AdvFaceGAN\Lib\site-packages\torchmetrics\utilities\prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(
C:\yy\installed_software\Anaconda3\envs\AdvFaceGAN\Lib\site-packages\torchmetrics\utilities\prints.py:70: FutureWarning: Importing `peak_signal_noise_ratio` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `peak_signal_noise_ratio` from `torchmetrics.image` instead.
  _future_warning(
1000it [00:42, 23.28it/s]


ArcFace  benchmark threshold_lfw:0.284027
ArcFace  benchmark rate:0.995000
ArcFace  before 1-FAR:0.997000
ArcFace  true pert:4.563634
ArcFace  mean source & target:0.5394 & 0.5479
ArcFace  average ssim:0.901691
ArcFace  average psnr:32.270766
ArcFace  average mse:35.989565
ArcFace  attack success rate1:0.998000
ArcFace  attack success rate2:0.997000
-----------------start white evaluate target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------



1000it [00:37, 26.66it/s]


ArcFace  benchmark threshold_lfw:0.284027
ArcFace  benchmark rate:0.995000
ArcFace  before 1-FAR:0.997000
ArcFace  true pert:4.584163
ArcFace  mean source & target:0.1394 & 0.9065
ArcFace  average ssim:0.894938
ArcFace  average psnr:32.212934
ArcFace  average mse:36.311438
ArcFace  attack success rate1:1.000000
ArcFace  attack success rate2:0.063000
-----------------start white evaluate target method data/CW_ArcFace_lfw_eps16_tpert1-------------------



...

ResNet50  benchmark threshold_lfw:0.191165
ResNet50  benchmark rate:0.997167
ResNet50  before 1-FAR:0.999000
ResNet50  true pert:4.031533
ResNet50  mean source & target:0.3327 & 0.4147
ResNet50  average ssim:0.937520
ResNet50  average psnr:33.257132
ResNet50  average mse:28.087230
ResNet50  attack success rate1:0.959000
ResNet50  attack success rate2:0.840000
-----------------start white evaluate target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------



1000it [00:41, 24.36it/s]


ResNet50  benchmark threshold_lfw:0.191165
ResNet50  benchmark rate:0.997167
ResNet50  before 1-FAR:0.999000
ResNet50  true pert:5.038827
ResNet50  mean source & target:0.2537 & 0.4612
ResNet50  average ssim:0.909595
ResNet50  average psnr:31.328965
ResNet50  average mse:43.875920
ResNet50  attack success rate1:0.979000
ResNet50  attack success rate2:0.671000
-----------------start white evaluate target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------



1000it [00:41, 24.23it/s]

ResNet50  benchmark threshold_lfw:0.191165
ResNet50  benchmark rate:0.997167
ResNet50  before 1-FAR:0.999000
ResNet50  true pert:4.642520
ResNet50  mean source & target:0.3477 & 0.4435
ResNet50  average ssim:0.939726
ResNet50  average psnr:32.030321
ResNet50  average mse:37.353922
ResNet50  attack success rate1:0.966000
ResNet50  attack success rate2:0.886000


In [3]:
# 表格打印
print("asr1_white:")
for col in zip(*asr1_white): 
    print("\t".join("{:.0f}%".format(x * 100) if x.is_integer() else "{:.1f}%".format(x * 100) for x in col))  
print("asr2_white:")
for col in zip(*asr2_white):
    print("\t".join("{:.0f}%".format(x * 100) if x.is_integer() else "{:.1f}%".format(x * 100) for x in col))

asr1_white:
99.8%	85.4%	96.1%
100%	100%	100%
100%	100%	100%
25.0%	41.9%	31.7%
62.5%	84.7%	94.2%
86.1%	89.9%	78.1%
100%	99.8%	100%
100%	100%	100%
58.1%	36.7%	48.8%
95.7%	93.4%	95.9%
97.7%	97.8%	97.9%
96.6%	94.8%	96.6%
asr2_white:
99.7%	70.6%	96.1%
6.3%	1.9%	37.8%
100%	98.4%	100%
25.0%	41.9%	31.7%
60.7%	20.8%	82.1%
83.5%	58.8%	66.6%
0%	0%	0%
0.9%	2.0%	2.5%
35.9%	11.7%	30.5%
95.2%	73.9%	84.0%
94.5%	55.8%	67.1%
96.2%	76.2%	88.6%


In [4]:
# BlackBox evaluation 单个替代模型评估 取三种白盒替代模型下最好的结果
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os

test_Blackbox_model_name_list = ['MobileFace','SphereFace','CosFace']
test_whitebox_model_name_list = ['ArcFace','FaceNet-casia','ResNet50']
asr1_black=np.empty((len(test_Blackbox_model_name_list), 12, len(test_whitebox_model_name_list)))
asr2_black=np.empty((len(test_Blackbox_model_name_list), 12, len(test_whitebox_model_name_list)))
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------\n".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    for idxz,model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------with white model {0}-------------------\n".format(model_name))
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/TIPIM_{model_name}",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------target method {0}-------------------".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for adv_pair in tqdm(subdirectories):
                test_transforms = torchvision.transforms.Compose([
                    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
                ])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                
                # 计算视觉指标
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
                
            print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
            print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
            print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
            print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
            print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))
            asr1_black[idxx][idxy][idxz]=np.mean(np.array(target_simi_scores) > th)
            asr2_black[idxx][idxy][idxz]=np.mean(np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))

-----------------start Black evaluate MobileFace-------------------

Load existing checkpoint
-----------------with white model ArcFace-------------------

-----------------target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:43<00:00, 23.00it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.997000
MobileFace  true pert:4.563634
MobileFace  mean source & target:0.5914 & 0.2430
MobileFace  average ssim:0.901691
MobileFace  average psnr:32.270766
MobileFace  average mse:35.989565
MobileFace  attack success rate1:0.648000
MobileFace  attack success rate2:0.648000
-----------------target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:44<00:00, 22.44it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.997000
MobileFace  true pert:4.584163
MobileFace  mean source & target:0.4358 & 0.3969
MobileFace  average ssim:0.894938
MobileFace  average psnr:32.212934
MobileFace  average mse:36.311438
MobileFace  attack success rate1:0.976000
MobileFace  attack success rate2:0.966000
-----------------target method data/CW_ArcFace_lfw_eps16_tpert1-------------------


...

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.031533
CosFace  mean source & target:0.5475 & 0.4060
CosFace  average ssim:0.937520
CosFace  average psnr:33.257132
CosFace  average mse:28.087230
CosFace  attack success rate1:0.923000
CosFace  attack success rate2:0.916000
-----------------target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [00:25<00:00, 39.53it/s]


CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:5.038827
CosFace  mean source & target:0.4605 & 0.4640
CosFace  average ssim:0.909595
CosFace  average psnr:31.328965
CosFace  average mse:43.875920
CosFace  attack success rate1:0.968000
CosFace  attack success rate2:0.920000
-----------------target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [00:26<00:00, 37.46it/s]

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.642520
CosFace  mean source & target:0.5529 & 0.4336
CosFace  average ssim:0.939726
CosFace  average psnr:32.030321
CosFace  average mse:37.353922
CosFace  attack success rate1:0.944000
CosFace  attack success rate2:0.939000


In [5]:
# 直接出表格
print("asr1_black:")
for row in np.max(asr1_black, axis=2).T:  
    print("\t".join("{:.0f}%".format(x * 100) if x * 100 == int(x * 100) else "{:.1f}%".format(x * 100) for x in row))
print("asr2_black:")
for row in np.max(asr2_black, axis=2).T:  
    print("\t".join("{:.0f}%".format(x * 100) if x * 100 == int(x * 100) else "{:.1f}%".format(x * 100) for x in row))

asr1_black:
64.8%	44.3%	40.8%
97.6%	86%	80.2%
7.1%	9.1%	8.1%
20.1%	37.1%	28.0%
88.6%	41.7%	67%
95.6%	84.5%	73.9%
89.1%	82.4%	91%
99.2%	83.2%	92.7%
65.6%	52.4%	38.2%
99.2%	79.5%	92.3%
99.3%	87.2%	96.8%
99.2%	83.5%	94.4%
asr2_black:
64.8%	44.3%	40.8%
96.6%	84.5%	78.9%
7.1%	9.1%	8.1%
20.1%	37.1%	28.0%
85.9%	40.4%	66.9%
33.7%	25.5%	27.5%
83.1%	76.7%	87.4%
75.4%	72.5%	83.1%
57.2%	42.2%	24.3%
92.7%	78.4%	91.6%
79%	82.3%	92%
96%	81.8%	93.9%


In [6]:
# BlackBox evaluation 
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os

test_Blackbox_model_name_list = ['MobileFace','SphereFace','CosFace']
asr1_black=np.empty((len(test_Blackbox_model_name_list), 6))
asr2_black=np.empty((len(test_Blackbox_model_name_list), 6))
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    # 设置对抗样本目录路径
    adv_samples_dirs = [
        f"data/DiffAM",
        f"data/AdvFace_lfw_eps8_tpert5.7",
        f"data/AdvMakeUP_lfw_tpert5.2",
        f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
        f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
        f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
    ]
    for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
        # 列出目录中的所有文件和文件夹
        all_files_and_dirs = os.listdir(adv_samples_dir)
        # 过滤出所有子文件夹
        subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
        ssim_scores = []
        psnr_scores = []
        mse_scores = []
        true_perts_score = []
        FAR_simi_scores = []
        source_simi_scores = []
        target_simi_scores = []
        print("-----------------start Black evaluate {0}-------------------".format(adv_samples_dir))
        for adv_pair in tqdm(subdirectories):
            test_transforms = torchvision.transforms.Compose([
                torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
            ])
            fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
            source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
            target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
            
            # 计算视觉指标
            true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
            ssim_scores.append(ssim(source_face, fake_after).item())
            psnr_scores.append(psnr(source_face, fake_after).item())
            mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
            
            # extract face embedding
            emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
            emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
            emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
            #  evaluation cosine similarity
            FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
            source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
            target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
            
        print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
        print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
        print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
        print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
        print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
        print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
        print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
        print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
        print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
        print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))
        asr1_black[idxx][idxy]=np.mean(np.array(target_simi_scores) > th)
        asr2_black[idxx][idxy]=np.mean(np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))


-----------------start Black evaluate MobileFace-------------------
Load existing checkpoint
-----------------start Black evaluate data/DiffAM-------------------


100%|██████████| 1000/1000 [00:53<00:00, 18.55it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.931000
MobileFace  true pert:27.928455
MobileFace  mean source & target:0.3231 & 0.2492
MobileFace  average ssim:0.824288
MobileFace  average psnr:16.852215
MobileFace  average mse:1366.958124
MobileFace  attack success rate1:0.656000
MobileFace  attack success rate2:0.572000
-----------------start Black evaluate data/AdvFace_lfw_eps8_tpert5.7-------------------


100%|██████████| 1000/1000 [00:47<00:00, 20.95it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.995000
MobileFace  true pert:5.737006
MobileFace  mean source & target:0.3874 & 0.3464
MobileFace  average ssim:0.915720
MobileFace  average psnr:30.463868
MobileFace  average mse:59.623360
MobileFace  attack success rate1:0.891000
MobileFace  attack success rate2:0.831000
-----------------start Black evaluate data/AdvMakeUP_lfw_tpert5.2-------------------


...

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.031533
CosFace  mean source & target:0.5475 & 0.4060
CosFace  average ssim:0.937520
CosFace  average psnr:33.257132
CosFace  average mse:28.087230
CosFace  attack success rate1:0.923000
CosFace  attack success rate2:0.916000
-----------------start Black evaluate data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [00:24<00:00, 41.09it/s]


CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:5.038827
CosFace  mean source & target:0.4605 & 0.4640
CosFace  average ssim:0.909595
CosFace  average psnr:31.328965
CosFace  average mse:43.875920
CosFace  attack success rate1:0.968000
CosFace  attack success rate2:0.920000
-----------------start Black evaluate data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [00:24<00:00, 40.88it/s]

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.642520
CosFace  mean source & target:0.5529 & 0.4336
CosFace  average ssim:0.939726
CosFace  average psnr:32.030321
CosFace  average mse:37.353922
CosFace  attack success rate1:0.944000
CosFace  attack success rate2:0.939000


In [7]:
# 表格打印
print("asr1_black:")
for col in zip(*asr1_black): 
    print("\t".join("{:.0f}%".format(x * 100) if x.is_integer() else "{:.1f}%".format(x * 100) for x in col))  
print("asr2_black:")
for col in zip(*asr2_black):
    print("\t".join("{:.0f}%".format(x * 100) if x.is_integer() else "{:.1f}%".format(x * 100) for x in col))

asr1_black:
65.6%	52.4%	38.2%
89.1%	82.4%	91.0%
20.1%	37.1%	28.0%
99.2%	79.5%	92.3%
99.3%	87.2%	96.8%
99.2%	83.5%	94.4%
asr2_black:
57.2%	42.2%	24.3%
83.1%	76.7%	87.4%
20.1%	37.1%	28.0%
92.7%	78.4%	91.6%
79.0%	82.3%	92.0%
96.0%	81.8%	93.9%


In [8]:
# BlackBox evaluation 单个替代模型评估 取三种白盒替代模型下最好的结果
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os
from PIL import Image
import torch
import numpy as np

test_Blackbox_model_name_list = ['MobileFace']
test_whitebox_model_name_list = ['ArcFace']
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------\n".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    for idxz,model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------with white model {0}-------------------\n".format(model_name))
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/TIPIM_{model_name}",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------target method {0}-------------------".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for adv_pair in tqdm(subdirectories):
                test_transforms = torchvision.transforms.Compose([
                    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
                ])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                
                # 计算视觉指标
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
                
            print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
            print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
            print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
            print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
            print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))


-----------------start Black evaluate MobileFace-------------------

Load existing checkpoint
-----------------with white model ArcFace-------------------

-----------------target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:38<00:00, 25.67it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.997000
MobileFace  true pert:4.563634
MobileFace  mean source & target:0.5914 & 0.2430
MobileFace  average ssim:0.901691
MobileFace  average psnr:32.270766
MobileFace  average mse:35.989565
MobileFace  attack success rate1:0.648000
MobileFace  attack success rate2:0.648000
-----------------target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:38<00:00, 25.96it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.997000
MobileFace  true pert:4.584163
MobileFace  mean source & target:0.4358 & 0.3969
MobileFace  average ssim:0.894938
MobileFace  average psnr:32.212934
MobileFace  average mse:36.311438
MobileFace  attack success rate1:0.976000
MobileFace  attack success rate2:0.966000
-----------------target method data/CW_ArcFace_lfw_eps16_tpert1-------------------


...

MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.994000
MobileFace  true pert:4.031533
MobileFace  mean source & target:0.3827 & 0.4956
MobileFace  average ssim:0.937520
MobileFace  average psnr:33.257132
MobileFace  average mse:28.087230
MobileFace  attack success rate1:0.992000
MobileFace  attack success rate2:0.927000
-----------------target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [00:45<00:00, 22.16it/s]


MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.994000
MobileFace  true pert:5.038827
MobileFace  mean source & target:0.3018 & 0.5429
MobileFace  average ssim:0.909595
MobileFace  average psnr:31.328965
MobileFace  average mse:43.875920
MobileFace  attack success rate1:0.993000
MobileFace  attack success rate2:0.790000
-----------------target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [00:45<00:00, 22.19it/s]

MobileFace  benchmark threshold_lfw:0.211167
MobileFace  benchmark rate:0.994500
MobileFace  before 1-FAR:0.994000
MobileFace  true pert:4.642520
MobileFace  mean source & target:0.4088 & 0.5214
MobileFace  average ssim:0.939726
MobileFace  average psnr:32.030321
MobileFace  average mse:37.353922
MobileFace  attack success rate1:0.992000
MobileFace  attack success rate2:0.960000


In [12]:
# BlackBox evaluation 单个替代模型评估 取三种白盒替代模型下最好的结果
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os

test_Blackbox_model_name_list = ['CosFace']
test_whitebox_model_name_list = ['Facenet-casia']
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------\n".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    for idxz,model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------with white model {0}-------------------\n".format(model_name))
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/TIPIM_{model_name}",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------target method {0}-------------------".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for adv_pair in tqdm(subdirectories):
                test_transforms = torchvision.transforms.Compose([
                    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
                ])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                
                # 计算视觉指标
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
                
            print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
            print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
            print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
            print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
            print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))


-----------------start Black evaluate CosFace-------------------

Load existing checkpoint
-----------------with white model Facenet-casia-------------------

-----------------target method data/FGSM_Facenet-casia_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:24<00:00, 40.66it/s]


CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.979000
CosFace  true pert:4.068449
CosFace  mean source & target:0.6460 & 0.1799
CosFace  average ssim:0.883741
CosFace  average psnr:32.411345
CosFace  average mse:35.911949
CosFace  attack success rate1:0.293000
CosFace  attack success rate2:0.293000
-----------------target method data/MIM_Facenet-casia_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:23<00:00, 42.22it/s]


CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.979000
CosFace  true pert:4.065439
CosFace  mean source & target:0.5352 & 0.3119
CosFace  average ssim:0.870401
CosFace  average psnr:32.307280
CosFace  average mse:36.316291
CosFace  attack success rate1:0.695000
CosFace  attack success rate2:0.692000
-----------------target method data/CW_Facenet-casia_lfw_eps16_tpert1-------------------


...

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.031533
CosFace  mean source & target:0.5475 & 0.4060
CosFace  average ssim:0.937520
CosFace  average psnr:33.257132
CosFace  average mse:28.087230
CosFace  attack success rate1:0.923000
CosFace  attack success rate2:0.916000
-----------------target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [00:26<00:00, 37.86it/s]


CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:5.038827
CosFace  mean source & target:0.4605 & 0.4640
CosFace  average ssim:0.909595
CosFace  average psnr:31.328965
CosFace  average mse:43.875920
CosFace  attack success rate1:0.968000
CosFace  attack success rate2:0.920000
-----------------target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [00:26<00:00, 37.65it/s]

CosFace  benchmark threshold_lfw:0.246258
CosFace  benchmark rate:0.986833
CosFace  before 1-FAR:0.943000
CosFace  true pert:4.642520
CosFace  mean source & target:0.5529 & 0.4336
CosFace  average ssim:0.939726
CosFace  average psnr:32.030321
CosFace  average mse:37.353922
CosFace  attack success rate1:0.944000
CosFace  attack success rate2:0.939000


In [10]:
# 鲁棒模型攻击评估
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os

test_Blackbox_model_name_list = ['IR50-Softmax','IR50-PGDSoftmax','IR50-TradesSoftmax','IR50-Softmax-BR','IR50-Softmax-RP','IR50-Softmax-JPEG']
test_whitebox_model_name_list = ['ArcFace']
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------\n".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    for idxz,model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------with white model {0}-------------------\n".format(model_name))
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/TIPIM_{model_name}",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------target method {0}-------------------".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for adv_pair in tqdm(subdirectories):
                test_transforms = torchvision.transforms.Compose([
                    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
                ])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                
                # 计算视觉指标
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
                
            print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
            print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
            print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
            print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
            print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))


-----------------start Black evaluate IR50-Softmax-------------------

Load existing checkpoint
-----------------with white model ArcFace-------------------

-----------------target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:28<00:00, 35.05it/s]


IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.563634
IR50-Softmax  mean source & target:0.6803 & 0.3314
IR50-Softmax  average ssim:0.901691
IR50-Softmax  average psnr:32.270766
IR50-Softmax  average mse:35.989565
IR50-Softmax  attack success rate1:0.443000
IR50-Softmax  attack success rate2:0.443000
-----------------target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:28<00:00, 34.83it/s]


IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.584163
IR50-Softmax  mean source & target:0.5216 & 0.5116
IR50-Softmax  average ssim:0.894938
IR50-Softmax  average psnr:32.212934
IR50-Softmax  average mse:36.311438
IR50-Softmax  attack success rate1:0.890000
IR50-Softmax  attack success rate2:0.844000
-----------------target method data/CW_ArcFace_lfw_eps16_tpert1-------------------


...

IR50-Softmax-JPEG  benchmark threshold_lfw:0.341502
IR50-Softmax-JPEG  benchmark rate:0.995833
IR50-Softmax-JPEG  before 1-FAR:0.991000
IR50-Softmax-JPEG  true pert:4.031533
IR50-Softmax-JPEG  mean source & target:0.4252 & 0.5727
IR50-Softmax-JPEG  average ssim:0.937520
IR50-Softmax-JPEG  average psnr:33.257132
IR50-Softmax-JPEG  average mse:28.087230
IR50-Softmax-JPEG  attack success rate1:0.952000
IR50-Softmax-JPEG  attack success rate2:0.663000
-----------------target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [01:09<00:00, 14.47it/s]


IR50-Softmax-JPEG  benchmark threshold_lfw:0.341502
IR50-Softmax-JPEG  benchmark rate:0.995833
IR50-Softmax-JPEG  before 1-FAR:0.991000
IR50-Softmax-JPEG  true pert:5.038827
IR50-Softmax-JPEG  mean source & target:0.3358 & 0.6202
IR50-Softmax-JPEG  average ssim:0.909595
IR50-Softmax-JPEG  average psnr:31.328965
IR50-Softmax-JPEG  average mse:43.875920
IR50-Softmax-JPEG  attack success rate1:0.969000
IR50-Softmax-JPEG  attack success rate2:0.453000
-----------------target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [01:15<00:00, 13.16it/s]

IR50-Softmax-JPEG  benchmark threshold_lfw:0.341502
IR50-Softmax-JPEG  benchmark rate:0.995833
IR50-Softmax-JPEG  before 1-FAR:0.991000
IR50-Softmax-JPEG  true pert:4.642520
IR50-Softmax-JPEG  mean source & target:0.4353 & 0.6031
IR50-Softmax-JPEG  average ssim:0.939726
IR50-Softmax-JPEG  average psnr:32.030321
IR50-Softmax-JPEG  average mse:37.353922
IR50-Softmax-JPEG  attack success rate1:0.961000
IR50-Softmax-JPEG  attack success rate2:0.700000


In [11]:
# 鲁棒模型攻击评估
import torchvision
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from torchmetrics.functional import structural_similarity_index_measure as ssim
from torchmetrics.functional import peak_signal_noise_ratio as psnr
from tqdm import tqdm
import os

test_Blackbox_model_name_list = ['IR50-Softmax']
test_whitebox_model_name_list = ['ArcFace']
for idxx,blackbox_model_name in enumerate(test_Blackbox_model_name_list):
    print("-----------------start Black evaluate {0}-------------------\n".format(blackbox_model_name))
    th = threshold_lfw[blackbox_model_name]['cos']
    model, img_shape = getmodel(blackbox_model_name)
    for idxz,model_name in enumerate(test_whitebox_model_name_list):
        print("-----------------with white model {0}-------------------\n".format(model_name))
        # 设置对抗样本目录路径
        adv_samples_dirs = [
            f"data/FGSM_{model_name}_lfw_eps6_tpert4.4",
            f"data/MIM_{model_name}_lfw_eps6_tpert4.4",
            f"data/CW_{model_name}_lfw_eps16_tpert1",
            f"data/AdvMakeUP_lfw_tpert5.2",
            f"data/AT3D_{model_name}_eye_nose_lfw_eps5_tpert13.5",
            f"data/TIPIM_{model_name}",
            f"data/AdvFace_lfw_eps8_tpert5.7",
            f"data/SiblingAttack_{model_name}_lfw_eps0.15_tpert10.7",
            f"data/DiffAM",
            f"data/AdvFaceGAN_target 4 8白盒 无stloss 990_lfw_eps4_tpert_4",
            f"data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5",
            f"data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6",
        ]
        for idxy,adv_samples_dir in enumerate(adv_samples_dirs):
            print("-----------------target method {0}-------------------".format(adv_samples_dir))
            # 列出目录中的所有文件和文件夹
            all_files_and_dirs = os.listdir(adv_samples_dir)
            # 过滤出所有子文件夹
            subdirectories = [d for d in all_files_and_dirs if os.path.isdir(os.path.join(adv_samples_dir, d))]
            ssim_scores = []
            psnr_scores = []
            mse_scores = []
            true_perts_score = []
            FAR_simi_scores = []
            source_simi_scores = []
            target_simi_scores = []
            for adv_pair in tqdm(subdirectories):
                test_transforms = torchvision.transforms.Compose([
                    torchvision.transforms.ToTensor(),  # 图片转为Tensor [0,1]
                ])
                fake_after = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/adv.png").convert('RGB')).unsqueeze(0).cuda()
                source_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/source.png").convert('RGB')).unsqueeze(0).cuda()
                target_face = test_transforms(Image.open(f"{adv_samples_dir}/{adv_pair}/target.png").convert('RGB')).unsqueeze(0).cuda()
                
                # 计算视觉指标
                true_perts_score.append(torch.norm(F.interpolate(fake_after, size=(112,112), mode='bilinear') - F.interpolate(source_face, size=(112,112), mode='bilinear')).item())
                ssim_scores.append(ssim(source_face, fake_after).item())
                psnr_scores.append(psnr(source_face, fake_after).item())
                mse_scores.append(F.mse_loss(source_face*255, fake_after*255).item())
                
                # extract face embedding
                emb_source = model.forward(F.interpolate(source_face*255, size=img_shape, mode='bilinear'))
                emb_target = model.forward(F.interpolate(target_face*255, size=img_shape, mode='bilinear'))
                emb_fake_after = model.forward(F.interpolate(fake_after*255, size=img_shape, mode='bilinear'))
                #  evaluation cosine similarity
                FAR_simi_scores.extend(torch.cosine_similarity(emb_source, emb_target).tolist())
                source_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_source).tolist())
                target_simi_scores.extend(torch.cosine_similarity(emb_fake_after, emb_target).tolist())
                
            print(blackbox_model_name, " benchmark threshold_lfw:%f" % threshold_lfw[blackbox_model_name]['cos'])
            print(blackbox_model_name, " benchmark rate:%f" % threshold_lfw[blackbox_model_name]['cos_acc'])
            print(blackbox_model_name, " before 1-FAR:%f" % np.mean(np.array(FAR_simi_scores) < th))
            print(blackbox_model_name, " true pert:%f" % np.mean(true_perts_score))
            print(blackbox_model_name, " mean source & target:%.4f & %.4f" % (np.mean(source_simi_scores), np.mean(target_simi_scores)))
            print(blackbox_model_name, " average ssim:%f" % np.mean(ssim_scores))
            print(blackbox_model_name, " average psnr:%f" % np.mean(psnr_scores))
            print(blackbox_model_name, " average mse:%f" % np.mean(mse_scores))
            print(blackbox_model_name, " attack success rate1:%f" % np.mean(np.array(target_simi_scores) > th))
            print(blackbox_model_name, " attack success rate2:%f" % np.mean((np.array(source_simi_scores) > th) & (np.array(target_simi_scores) > th)))


-----------------start Black evaluate IR50-Softmax-------------------

Load existing checkpoint
-----------------with white model ArcFace-------------------

-----------------target method data/FGSM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:30<00:00, 33.20it/s]


IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.563634
IR50-Softmax  mean source & target:0.6803 & 0.3314
IR50-Softmax  average ssim:0.901691
IR50-Softmax  average psnr:32.270766
IR50-Softmax  average mse:35.989565
IR50-Softmax  attack success rate1:0.443000
IR50-Softmax  attack success rate2:0.443000
-----------------target method data/MIM_ArcFace_lfw_eps6_tpert4.4-------------------


100%|██████████| 1000/1000 [00:31<00:00, 32.09it/s]


IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.584163
IR50-Softmax  mean source & target:0.5216 & 0.5116
IR50-Softmax  average ssim:0.894938
IR50-Softmax  average psnr:32.212934
IR50-Softmax  average mse:36.311438
IR50-Softmax  attack success rate1:0.890000
IR50-Softmax  attack success rate2:0.844000
-----------------target method data/CW_ArcFace_lfw_eps16_tpert1-------------------


...

IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.031533
IR50-Softmax  mean source & target:0.4203 & 0.5780
IR50-Softmax  average ssim:0.937520
IR50-Softmax  average psnr:33.257132
IR50-Softmax  average mse:28.087230
IR50-Softmax  attack success rate1:0.957000
IR50-Softmax  attack success rate2:0.646000
-----------------target method data/AdvFaceGAN_target 5 8白盒 无stloss 990_lfw_eps5_tpert5-------------------


100%|██████████| 1000/1000 [00:56<00:00, 17.80it/s]


IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:5.038827
IR50-Softmax  mean source & target:0.3308 & 0.6257
IR50-Softmax  average ssim:0.909595
IR50-Softmax  average psnr:31.328965
IR50-Softmax  average mse:43.875920
IR50-Softmax  attack success rate1:0.971000
IR50-Softmax  attack success rate2:0.438000
-----------------target method data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6-------------------


100%|██████████| 1000/1000 [00:56<00:00, 17.68it/s]

IR50-Softmax  benchmark threshold_lfw:0.342332
IR50-Softmax  benchmark rate:0.996000
IR50-Softmax  before 1-FAR:0.992000
IR50-Softmax  true pert:4.642520
IR50-Softmax  mean source & target:0.4336 & 0.6067
IR50-Softmax  average ssim:0.939726
IR50-Softmax  average psnr:32.030321
IR50-Softmax  average mse:37.353922
IR50-Softmax  attack success rate1:0.962000
IR50-Softmax  attack success rate2:0.694000


In [18]:
# 表现差的样本提取并整合可视化
import torch
import torchvision
from PIL import Image
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt

# 配置参数
adv_samples_dir = "data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6"
model_name = 'MobileFace'
th = threshold_lfw[model_name]['cos']
model, img_shape = getmodel(model_name)

# 创建存储目录
vis_dir = os.path.join("poor_samples")
os.makedirs(vis_dir, exist_ok=True)

# 收集符合条件样本的路径
qualified_samples = []
print("-----------------开始筛选失败样本-------------------")
all_dirs = [d for d in os.listdir(adv_samples_dir) if os.path.isdir(os.path.join(adv_samples_dir, d))]

for adv_pair in tqdm(all_dirs):
    # 加载图像
    source_path = f"{adv_samples_dir}/{adv_pair}/source.png"
    target_path = f"{adv_samples_dir}/{adv_pair}/target.png"
    adv_path = f"{adv_samples_dir}/{adv_pair}/adv.png"
    
    # 计算相似度
    source = Image.open(source_path).convert('RGB')
    target = Image.open(target_path).convert('RGB')
    adv = Image.open(adv_path).convert('RGB')
    
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    
    source_tensor = transform(source).unsqueeze(0).cuda()
    target_tensor = transform(target).unsqueeze(0).cuda()
    adv_tensor = transform(adv).unsqueeze(0).cuda()
    
    # 特征提取
    emb_source = model(F.interpolate(source_tensor*255, size=img_shape, mode='bilinear'))
    emb_target = model(F.interpolate(target_tensor*255, size=img_shape, mode='bilinear'))
    emb_adv = model(F.interpolate(adv_tensor*255, size=img_shape, mode='bilinear'))
    
    # 计算相似度
    source_sim = torch.cosine_similarity(emb_adv, emb_source).item()
    target_sim = torch.cosine_similarity(emb_adv, emb_target).item()
    
    # 筛选条件
    if source_sim > th and target_sim < th:
    # if target_sim > th and source_sim<th:
        qualified_samples.append({
            "id": adv_pair,
            "source": source,
            "target": target,
            "adv": adv,
            "source_sim": source_sim,
            "target_sim": target_sim
        })

# 创建组合大图
if qualified_samples:
    print(f"发现{len(qualified_samples)}个失败样本，生成组合视图...")
    
    # 动态计算布局
    n = len(qualified_samples)
    cols = 3  # 每行显示3个样本
    rows = (n + cols - 1) // cols
    
    # 创建画布
    fig = plt.figure(figsize=(20, 7*rows))  # 宽度固定，高度自适应
    plt.rcParams['axes.titlesize'] = 12
    plt.subplots_adjust(wspace=0.05, hspace=0.2)
    
    # 遍历样本绘制子图
    for idx, sample in enumerate(qualified_samples, 1):
        ax = fig.add_subplot(rows, cols, idx)
        
        # 创建三联子图
        plt.subplot(rows, cols*3, 3*idx-2)
        plt.imshow(sample['source'])
        plt.title(f"Source\nFSS={sample['source_sim']:.2f}", fontsize=10)
        plt.axis('off')
        
        plt.subplot(rows, cols*3, 3*idx-1)
        plt.imshow(sample['target'])
        plt.title(f"Target\nFTS={sample['target_sim']:.2f}", fontsize=10)
        plt.axis('off')
        
        plt.subplot(rows, cols*3, 3*idx)
        plt.imshow(sample['adv'])
        plt.title("Adversarial", fontsize=10)
        plt.axis('off')
    
    # 保存和显示
    plt.tight_layout()
    save_path = os.path.join(vis_dir, f"combined_failures_{len(qualified_samples)}samples.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"已保存组合视图到：{save_path}")
else:
    print("未发现符合要求的失败样本")

print("-----------------处理完成-------------------")

Load existing checkpoint
-----------------开始筛选失败样本-------------------


100%|██████████| 1000/1000 [03:03<00:00,  5.46it/s]


发现8个失败样本，生成组合视图...
已保存组合视图到：poor_samples\combined_failures_8samples.png
-----------------处理完成-------------------


In [31]:
# 表现好的样本提取并整合可视化
import torch
import torchvision
from PIL import Image
from fr_models.get_model import getmodel
from fr_models.config import threshold_lfw
import torch.nn.functional as F
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt

# 配置参数
adv_samples_dir = "data/AdvFaceGAN_target 5 8白盒 奇怪ssim 92ssim 双身份损失0.15 2490_lfw_eps5_tpert4.6"
model_name = 'MobileFace'
th = threshold_lfw[model_name]['cos']
model, img_shape = getmodel(model_name)

# 创建存储目录
vis_dir = os.path.join("poor_samples")
os.makedirs(vis_dir, exist_ok=True)

# 收集符合条件样本的路径
qualified_samples = []
print("-----------------开始筛选成功样本-------------------")
all_dirs = [d for d in os.listdir(adv_samples_dir) if os.path.isdir(os.path.join(adv_samples_dir, d))]

for adv_pair in tqdm(all_dirs):
    # 加载图像
    source_path = f"{adv_samples_dir}/{adv_pair}/source.png"
    target_path = f"{adv_samples_dir}/{adv_pair}/target.png"
    adv_path = f"{adv_samples_dir}/{adv_pair}/adv.png"
    
    # 计算相似度
    source = Image.open(source_path).convert('RGB')
    target = Image.open(target_path).convert('RGB')
    adv = Image.open(adv_path).convert('RGB')
    
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    
    source_tensor = transform(source).unsqueeze(0).cuda()
    target_tensor = transform(target).unsqueeze(0).cuda()
    adv_tensor = transform(adv).unsqueeze(0).cuda()
    
    # 特征提取
    emb_source = model(F.interpolate(source_tensor*255, size=img_shape, mode='bilinear'))
    emb_target = model(F.interpolate(target_tensor*255, size=img_shape, mode='bilinear'))
    emb_adv = model(F.interpolate(adv_tensor*255, size=img_shape, mode='bilinear'))
    
    # 计算相似度
    source_sim = torch.cosine_similarity(emb_adv, emb_source).item()
    target_sim = torch.cosine_similarity(emb_adv, emb_target).item()
    
    # 筛选条件
    if source_sim > 2.5*th and target_sim > 2.5*th:
    # if target_sim  th and source_sim<th:
        qualified_samples.append({
            "id": adv_pair,
            "source": source,
            "target": target,
            "adv": adv,
            "source_sim": source_sim,
            "target_sim": target_sim
        })

# 创建组合大图
if qualified_samples:
    print(f"发现{len(qualified_samples)}个成功样本，生成组合视图...")
    
    # 动态计算布局
    n = len(qualified_samples)
    cols = 3  # 每行显示3个样本
    rows = (n + cols - 1) // cols
    
    # 创建画布
    fig = plt.figure(figsize=(20, 7*rows))  # 宽度固定，高度自适应
    plt.rcParams['axes.titlesize'] = 12
    plt.subplots_adjust(wspace=0.05, hspace=0.2)
    
    # 遍历样本绘制子图
    for idx, sample in enumerate(qualified_samples, 1):
        ax = fig.add_subplot(rows, cols, idx)
        
        # 创建三联子图
        plt.subplot(rows, cols*3, 3*idx-2)
        plt.imshow(sample['source'])
        plt.title(f"Source\nFSS={sample['source_sim']:.2f}", fontsize=10)
        plt.axis('off')
        
        plt.subplot(rows, cols*3, 3*idx-1)
        plt.imshow(sample['target'])
        plt.title(f"Target\nFTS={sample['target_sim']:.2f}", fontsize=10)
        plt.axis('off')
        
        plt.subplot(rows, cols*3, 3*idx)
        plt.imshow(sample['adv'])
        plt.title("Adversarial", fontsize=10)
        plt.axis('off')
    
    # 保存和显示
    plt.tight_layout()
    save_path = os.path.join(vis_dir, f"combined_excellent_{len(qualified_samples)}samples.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"已保存组合视图到：{save_path}")
else:
    print("未发现符合要求的成功样本")

print("-----------------处理完成-------------------")

Load existing checkpoint
-----------------开始筛选成功样本-------------------


100%|██████████| 1000/1000 [03:03<00:00,  5.45it/s]


发现44个成功样本，生成组合视图...
已保存组合视图到：poor_samples\combined_excellent_44samples.png
-----------------处理完成-------------------


In [42]:
import random
from matplotlib.gridspec import GridSpec

# 随机选择样本（最多10个）
max_samples = 8
selected_samples = random.sample(qualified_samples, min(len(qualified_samples), max_samples))

# 创建专业排版画布
plt.figure(figsize=(18, 12), dpi=150)
gs = GridSpec(nrows=5,  # 每列显示5个样本
             ncols=6,  # 每个样本占3列（Source/Target/Adversarial）
             wspace=0.08, 
             hspace=0.3,
             width_ratios=[1,1,1]*2)  # 列宽比例

# 样式配置
title_style = {'fontsize': 14, 'y': 1.05, 'fontweight':'semibold'}
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'axes.edgecolor': '#444444',
    'axes.labelcolor': '#444444',
})

# 绘制每个样本
for idx, sample in enumerate(selected_samples):
    row = idx // 2  # 每行显示2个样本
    col = (idx % 2) * 3  # 每个样本占3列
    
    # Source图像
    ax0 = plt.subplot(gs[row, col])
    ax0.imshow(sample['source'])
    ax0.set_title(f"Source (FSS={sample['source_sim']:.2f})", **title_style)
    ax0.axis('off')
    
    # Target图像
    ax1 = plt.subplot(gs[row, col+1])
    ax1.imshow(sample['target'])
    ax1.set_title(f"Target (FTS={sample['target_sim']:.2f})", **title_style)
    ax1.axis('off')
    
    # Adversarial图像
    ax2 = plt.subplot(gs[row, col+2])
    im = ax2.imshow(sample['adv'])
    ax2.set_title("Adversarial", **title_style)
    ax2.axis('off')
    
    # # 添加色标
    # if idx == 0:  # 只在第一个样本添加色标
    #     cbar = plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    #     cbar.ax.tick_params(labelsize=8)

# # 添加全局标题
# plt.suptitle("Adversarial Attack Failure Cases Analysis\n(FSS > 0.8 & FTS < Threshold)", 
#              y=0.98, 
#              fontsize=14,
#              fontweight='bold')

# 保存高清图像
save_path = os.path.join(vis_dir, f"random_%d.png" % max_samples)
plt.savefig(save_path, bbox_inches='tight', pad_inches=0.2, facecolor='white')
plt.close()